In [1]:
import os
import sys
import time
import yaml
import pandas as pd
import json
from string import Template

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
DATA_PATH = local_config['DATA_PATH']

sys.path.append(os.path.join(LOCAL_PATH, "src/python"))

from utils import is_casenum, extract_json

In [2]:
meetings_df = pd.read_csv(os.path.join(DATA_PATH, "intermediate_data/cpc/meetings-manifest.csv"))
DATES = sorted(list(meetings_df['date']))

In [3]:
types_df = {}
actions_df = {}

for idx, row in meetings_df.iterrows():
    date = row['date']
    year = date[0:4]
    input_filepath = os.path.join(DATA_PATH, "intermediate_data/cpc", year, date, "agenda-cleaned-summarized-2.json")
    if not os.path.exists(input_filepath):
        continue
    with open(input_filepath, 'r', encoding='utf-8') as f:
        j = json.load(f)

    for item in j:
        project_type = item["ai_summary_2"].get("project_type", "")
        if len(project_type)==0:
            continue
        types_df[project_type] = types_df.get(project_type, 0) + 1

        requested_actions = item["ai_summary_2"].get("requested_actions", [])
        for action in requested_actions:
            actions_df[action] = actions_df.get(action, 0) + 1

types_df = pd.DataFrame(list(types_df.items()), columns=["type", "count"])
types_df = types_df.sort_values(by="count", ascending=False)
#types_df.to_csv(os.path.join(DATA_PATH, "intermediate_data/cpc/project_types.csv"), index=False)

actions_df = pd.DataFrame(list(actions_df.items()), columns=["action", "count"])
actions_df = actions_df.sort_values(by="count", ascending=False)
#actions_df.to_csv(os.path.join(DATA_PATH, "intermediate_data/cpc/requested_actions.csv"), index=False)



In [4]:
types_df.head(10)

,type,count
2,new residential development,207
0,new mixed-use residential and commercial devel...,99
8,new mixed-use development,70
38,new commercial development,44
33,general plan amendment,21
61,new mixed-use residential development,12
12,zoning code amendment,11
13,specific plan amendment,9
27,new hotel development,9
94,code amendment,7
